# 🎭 Texto → Animação 3D → UE5

Gere animações de personagem a partir de texto (ex.: *"a maid serves tea gracefully"*) e baixe
arquivos prontos para **Unreal Engine 5** (IK Retargeter → Manny). Roda no **Colab gratuito (T4)**.

## Como usar (4 passos, nesta ordem)

| # | Célula | O que faz |
|---|---|---|
| ① | **CONFIG** | escolha o modelo + escreva o prompt + ajustes (só esta célula tem campos) |
| ② | **biblioteca** | código de exportação (GLB/FBX/BVH) — nunca precisa mexer |
| ③ | **SETUP** | instala as dependências do modelo (1x por sessão, com cache) |
| ④ | **GENERATE** | gera as variações da animação |
| ⑤ | **EXPORT** | preview 3D + download do **GLB / FBX(UE5) / BVH / NPZ** |

> **Modo limpo:** as células de código abrem com o **código escondido** (`cellView: form`) — você vê
> só os campos, os logs e o preview. Para ver o código de qualquer célula, clique em **Mostrar código**.

## ⚠️ Antes de rodar (só para o Kimodo, o modelo padrão)

O Kimodo usa o **Llama-3-8B** como encoder de texto, e esse modelo é *gated* no Hugging Face:

1. Aceite a licença em <https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct> (mesma conta do Colab).
2. Crie um token de leitura em <https://huggingface.co/settings/tokens>.
3. No Colab: ícone de **chave** (🔑) na barra esquerda → **Secrets** → nome `HF_TOKEN` → valor = seu token.
   Marque *"Notebook access"*.

Sem isso o SETUP avisa e a geração falha ao baixar o encoder.

**Primeira geração é demorada:** o encoder de texto (Llama-3-8B, ~16 GB) é baixado 1x por runtime.
Depois fica no cache local (`/content/ai_mocap_cache/hf`).

## Encoder de texto: qual escolher (campo no CONFIG)

O Llama-3-8B em bf16 pesa ~16 GB — não cabe inteiro nem na VRAM (16 GB) nem na RAM (≈12,7 GB) do
Colab gratuito. Por isso o padrão é **auto**, que decide assim:

| Situação | Escolha | Onde roda |
|---|---|---|
| GPU com ≥ 20 GB de VRAM (A100/L4/4090…) | `gpu` | bf16 na GPU — o mais rápido |
| RAM ≥ 24 GB (Kaggle, Colab Pro) | `cpu` | bf16 na CPU — ~<3 GB de VRAM |
| **T4 grátis / pouca RAM** (o caso comum) | `4bit` | **quantizado em 4-bit na GPU (~6 GB)** |

Se estourar memória, mude para `4bit` (ou `cpu`), reduza **Variações** e **Duração** e rode ④ de novo.

## Modelos (todos gratuitos)

| Modelo | Qualidades | Memória | Licença |
|---|---|---|---|
| ⭐ **Kimodo SOMA-RP v1.1** (NVIDIA) | Melhor qualidade; **77 joints (dedos!)**; realista | ~6 GB VRAM (encoder 4-bit) | NVIDIA Open Model ✅ comercial |
| **HY-Motion 1.0 Lite** (Tencent) | Muito rápido; bom p/ dança e movimentos casuais | ~16 GB (cabe no T4) | Tencent Community ✅ comercial |
| **MoMask** (CVPR 2024) | Segue bem instruções precisas (contar passos, direções) | ~8 GB | MIT ✅ comercial |

## Como funciona a sessão

- **1 modelo por sessão** (de propósito). Para trocar: `Runtime → Restart runtime`.
- Reexecutar o ③ SETUP **não re-baixa** nada (cache em `/content`, e no Drive se marcado).
- Marque **"Google Drive"** no ① CONFIG para o cache dos repos/pesos sobreviver a resets.

## Saída para o UE

- `*_ue.fbx` — ossos **já batizados com os nomes do Manny** (`pelvis`, `spine_01`, `thigh_l`,
  `thumb_01_l`…) → o IK Retargeter mapeia quase tudo automático.
- `.glb` — mesma animação para Blender/preview. `.bvh` — Blender/Mixamo. `.npy` — dados brutos.

## 🔧 Se o ③ SETUP falhar

- O SETUP mostra o **fim do log** completo (`/content/ai_mocap_cache/setup.log`) em vez de só
  “falhou o comando”.
- O Kimodo é instalado **sem** compilar o pós-processamento em C++ (é o que quebrava o `pip install`
  no Colab, por falta de `cmake`). A geração então roda com `--no-postprocess`.
- Quer o pós-processamento (limpa deslize dos pés)? Marque **"Compilar pos-processamento"** no
  ① CONFIG — o setup instala `cmake`/`eigen`/`pybind11` e compila (~5 min).


In [ ]:
#@title ① CONFIG — modelo + prompt + opções
# ════════════════════════ ① CONFIG ════════════════════════
# 1) Escolha o MODELO desta sessão (para trocar: Runtime -> Restart)
# 2) Escreva o PROMPT (em inglês, descrevendo o movimento)
# 3) Rode esta célula -> depois rode ③ SETUP (1a vez) e ④ GENERATE

import ipywidgets as widgets
import datetime

MODELS = {
    "kimodo":   "Kimodo SOMA-RP v1.1 (NVIDIA) - MELHOR qualidade, 77 joints (dedos!), ~6GB VRAM",
    "hymotion": "HY-Motion 1.0 Lite (Tencent) - rapido, bom p/ danca e movimentos casuais",
    "momask":   "MoMask (CVPR 2024) - segue bem instrucoes precisas",
}

# Onde o encoder de texto do Kimodo (Llama-3-8B) roda. So' afeta o Kimodo.
ENCODERS = {
    "auto": "auto - escolhe sozinho pela VRAM/RAM (recomendado)",
    "4bit": "GPU 4-bit - Llama-3-8B quantizado (~6GB VRAM) - cabe no T4 gratis",
    "gpu":  "GPU bf16 - o mais rapido (precisa de ~20GB de VRAM)",
    "cpu":  "CPU bf16 - precisa de ~20GB de RAM (Kaggle / Colab Pro)",
}

# o CONFIG pode ser re-executado so' p/ mudar o prompt: nao zera o historico.
try:
    STATE
except NameError:
    STATE = {"model": None, "loaded": False, "weights_ready": False, "motions": []}

_model_dd   = widgets.Dropdown(options=list(MODELS), value="kimodo",
                               description="MODELO (sessao)", layout=widgets.Layout(width="700px"))
_prompt_tb  = widgets.Textarea(
    value="A cute maid bows politely, then picks up a teacup and serves it with both hands",
    description="PROMPT (em ingles - descreva o movimento)", rows=3,
    layout=widgets.Layout(width="700px"))
_row1 = widgets.HBox([
    widgets.IntSlider(value=5, min=2, max=10, step=1, description="Duracao (s)"),
    widgets.IntSlider(value=2, min=1, max=4, description="Variacoes"),
    widgets.IntText(value=42, description="Seed"),
])
_row2 = widgets.HBox([
    widgets.Dropdown(options=["ue5_manny", "original"], value="ue5_manny",
                     description="Ossos no arquivo", layout=widgets.Layout(width="320px")),
    widgets.Dropdown(options=["floor", "origin", "inplace"], value="floor",
                     description="Modo do root (UE)", layout=widgets.Layout(width="300px")),
    widgets.FloatText(value=0.95, description="Altura do pelvis (m)",
                      layout=widgets.Layout(width="260px")),
])
_row3 = widgets.HBox([
    widgets.Dropdown(options=list(ENCODERS), value="auto",
                     description="Encoder de texto (Kimodo)",
                     layout=widgets.Layout(width="430px")),
    widgets.Checkbox(value=False,
                     description="Compilar pos-processamento foot-skate (+~5min no setup)",
                     layout=widgets.Layout(width="430px")),
])
_drive_cb = widgets.Checkbox(value=False,
                             description="Manter cache de pesos no Google Drive (recomendado - nao precisa remountar)")

CFG = {}

def _run_config():
    CFG.update({
        "model": _model_dd.value,
        "prompt": _prompt_tb.value.strip(),
        "duration": int(_row1.children[0].value),
        "samples": int(_row1.children[1].value),
        "seed": int(_row1.children[2].value),
        "skeleton": _row2.children[0].value,
        "root_mode": _row2.children[1].value,
        "pelvis_h": float(_row2.children[2].value),
        "use_drive": _drive_cb.value,
        "text_encoder": _row3.children[0].value,
        "compile_postprocess": bool(_row3.children[1].value),
        "_ok": True,
    })
    print("=" * 74)
    print(f"MODELO (esta sessao): {CFG['model']}")
    print(f"                     {MODELS[CFG['model']]}")
    print(f"PROMPT:      {CFG['prompt']}")
    print(f"DURACAO:     {CFG['duration']}s | VARIACOES: {CFG['samples']} | SEED: {CFG['seed']}")
    print(f"OSSOS:       {CFG['skeleton']}  (ue5_manny = ossos batizados p/ Manny)")
    print(f"ROOT (UE):   {CFG['root_mode']}  (floor = pelvis a {CFG['pelvis_h']}m do chao)")
    if CFG["model"] == "kimodo":
        print(f"ENCODER:     {CFG['text_encoder']}  ({ENCODERS[CFG['text_encoder']]})")
        print(f"FOOT-SKATE:  {'vai compilar o C++ no setup' if CFG['compile_postprocess'] else 'desligado (setup rapido)'}")
    print("-" * 74)
    if STATE["loaded"] and STATE["model"] == CFG["model"]:
        print(f"  {CFG['model']} JA PRONTO nesta sessao.")
        print("  -> NAO vai baixar nada de novo. Rode direto a celula ④ GENERATE.")
        if CFG["model"] == "kimodo":
            enc = STATE.get("text_encoder", "auto")
            print(f"  -> encoder de texto: {enc} (recarrega do cache local a cada geracao)")
    elif STATE["model"] is not None and STATE["model"] != CFG["model"]:
        print(f"  ATT: a sessao esta presa ao modelo '{STATE['model']}'.")
        print("  Para trocar: Runtime -> Restart runtime (os pesos ficam em cache/Drive).")
    else:
        print("  1o uso deste modelo na sessao: rode a celula ③ SETUP")
        print("  (instala dependencias + baixa pesos - com cache, uma unica vez).")
    print("=" * 74)

_run_config()
display(widgets.VBox([_model_dd, _prompt_tb, _row1, _row2, _row3, _drive_cb]))

In [ ]:
# ════════════════════════ BIBLIOTECA ════════════════════════
# Converters SMPL/SOMA -> GLB / FBX(UE5) / BVH + FK + preview.
# (codigo fixo - rode como esta)

# -*- coding: utf-8 -*-
"""
ai_mocap_lib — Text-to-Motion -> GLB / FBX(UE5) / BVH exporters.
Only dependency: numpy.
"""
import json
import math
import re
import struct
import numpy as np

# =====================================================================
# 1. MATH
# =====================================================================
def _aa_to_mat_single(aa):
    aa = np.asarray(aa, dtype=np.float64)
    n = np.linalg.norm(aa)
    if n < 1e-12:
        return np.eye(3)
    a = aa / n
    t = n
    K = np.array([[0.0, -a[2], a[1]], [a[2], 0.0, -a[0]], [-a[1], a[0], 0.0]])
    return np.eye(3) + np.sin(t) * K + (1.0 - np.cos(t)) * (K @ K)

def aa_to_mat(aa):
    aa = np.asarray(aa, dtype=np.float64)
    orig = aa.shape
    single = orig == (3,)
    aa = aa.reshape(-1, 3)
    out = np.stack([_aa_to_mat_single(a) for a in aa])
    return out[0] if single else out.reshape(orig[:-1] + (3, 3))

def _mat_to_quat_single(R):
    R = np.asarray(R, dtype=np.float64)
    tr = R[0, 0] + R[1, 1] + R[2, 2]
    if tr > 0.0:
        s = math.sqrt(tr + 1.0) * 2.0
        w = 0.25 * s
        x = (R[2, 1] - R[1, 2]) / s
        y = (R[0, 2] - R[2, 0]) / s
        z = (R[1, 0] - R[0, 1]) / s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = math.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2.0
        w = (R[2, 1] - R[1, 2]) / s
        x = 0.25 * s
        y = (R[0, 1] + R[1, 0]) / s
        z = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = math.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2.0
        w = (R[0, 2] - R[2, 0]) / s
        x = (R[0, 1] + R[1, 0]) / s
        y = 0.25 * s
        z = (R[1, 2] + R[2, 1]) / s
    else:
        s = math.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2.0
        w = (R[1, 0] - R[0, 1]) / s
        x = (R[0, 2] + R[2, 0]) / s
        y = (R[1, 2] + R[2, 1]) / s
        z = 0.25 * s
    q = np.array([x, y, z, w], dtype=np.float64)
    return q / np.linalg.norm(q)

def mat_to_quat(R):
    R = np.asarray(R, dtype=np.float64)
    orig = R.shape
    single = orig == (3, 3)
    R = R.reshape(-1, 3, 3)
    out = np.stack([_mat_to_quat_single(m) for m in R])
    return out[0] if single else out.reshape(orig[:-2] + (4,))

def quat_to_mat(q):
    q = np.asarray(q, dtype=np.float64)
    single = q.shape == (4,)
    q = q.reshape(-1, 4)
    n = np.linalg.norm(q, axis=1, keepdims=True)
    n[n == 0] = 1.0
    q = q / n
    x, y, z, w = q.T
    out = np.stack([
        np.stack([1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)], axis=-1),
        np.stack([2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)], axis=-1),
        np.stack([2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)], axis=-1),
    ], axis=1)
    return out[0] if single else out.reshape(q.shape[:-1] + (3, 3))

def normalize_quats(q):
    q = np.asarray(q, dtype=np.float64)
    n = np.linalg.norm(q, axis=-1, keepdims=True)
    n[n == 0] = 1.0
    return q / n

# =====================================================================
# 2. SKELETONS
# =====================================================================
SMPL22_NAMES = ["pelvis", "left_hip", "right_hip", "spine_1", "left_knee",
                "right_knee", "spine_2", "left_ankle", "right_ankle",
                "spine_3", "left_foot", "right_foot", "neck_1", "head",
                "left_collar", "right_collar", "left_shoulder",
                "right_shoulder", "left_elbow", "right_elbow", "left_wrist",
                "right_wrist"]
SMPL22_PARENTS = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19]
SMPL22_OFFSETS_M = [
    [0.0, 0.0, 0.0],
    [0.069520, -0.091406, -0.006815],
    [-0.067670, -0.090522, -0.004320],
    [-0.002533, 0.108963, -0.026696],
    [0.034277, -0.375199, -0.004496],
    [-0.038290, -0.382569, -0.008850],
    [0.005487, 0.135180, 0.001092],
    [-0.013596, -0.397960, -0.043693],
    [0.015774, -0.398415, -0.042312],
    [0.001457, 0.052922, 0.025425],
    [0.026358, -0.055791, 0.119288],
    [-0.025372, -0.048144, 0.123348],
    [-0.002778, 0.213870, -0.042857],
    [0.078845, 0.121749, -0.034090],
    [-0.081759, 0.118833, -0.038615],
    [0.005152, 0.064970, 0.051349],
    [0.090977, 0.030469, -0.008868],
    [-0.096012, 0.032551, -0.009143],
    [0.259612, -0.012772, -0.027456],
    [-0.253742, -0.013329, -0.021401],
    [0.249234, 0.008986, -0.001171],
    [-0.255298, 0.007772, -0.005559],
]
SMPL22_ENDS = [("L_wrist_end", 20, [0.084042, -0.008162, -0.014945]),
               ("R_wrist_end", 21, [-0.084622, -0.006117, -0.010315])]

UE5_MAP_SMPL22 = {
    "pelvis": "pelvis",
    "left_hip": "thigh_l", "right_hip": "thigh_r",
    "left_knee": "calf_l", "right_knee": "calf_r",
    "left_ankle": "foot_l", "right_ankle": "foot_r",
    "left_foot": "ball_l", "right_foot": "ball_r",
    "spine_1": "spine_01", "spine_2": "spine_02", "spine_3": "spine_03",
    "neck_1": "neck_01", "head": "head",
    "left_collar": "clavicle_l", "right_collar": "clavicle_r",
    "left_shoulder": "upperarm_l", "right_shoulder": "upperarm_r",
    "left_elbow": "lowerarm_l", "right_elbow": "lowerarm_r",
    "left_wrist": "hand_l", "right_wrist": "hand_r",
}
UE5_MAP_SOMA77 = {
    "Hips": "pelvis",
    "Spine1": "spine_01", "Spine2": "spine_02", "Chest": "spine_03",
    "Neck1": "neck_01", "Neck2": "neck_02", "Head": "head",
    "LeftShoulder": "clavicle_l", "RightShoulder": "clavicle_r",
    "LeftArm": "upperarm_l", "RightArm": "upperarm_r",
    "LeftForeArm": "lowerarm_l", "RightForeArm": "lowerarm_r",
    "LeftHand": "hand_l", "RightHand": "hand_r",
    "LeftLeg": "thigh_l", "RightLeg": "thigh_r",
    "LeftShin": "calf_l", "RightShin": "calf_r",
    "LeftFoot": "foot_l", "RightFoot": "foot_r",
    "LeftToeBase": "ball_l", "RightToeBase": "ball_r",
    "LeftHandThumb1": "thumb_01_l", "LeftHandThumb2": "thumb_02_l", "LeftHandThumb3": "thumb_03_l",
    "RightHandThumb1": "thumb_01_r", "RightHandThumb2": "thumb_02_r", "RightHandThumb3": "thumb_03_r",
    "LeftHandIndex1": "index_01_l", "LeftHandIndex2": "index_02_l", "LeftHandIndex3": "index_03_l", "LeftHandIndex4": "index_04_l",
    "RightHandIndex1": "index_01_r", "RightHandIndex2": "index_02_r", "RightHandIndex3": "index_03_r", "RightHandIndex4": "index_04_r",
    "LeftHandMiddle1": "middle_01_l", "LeftHandMiddle2": "middle_02_l", "LeftHandMiddle3": "middle_03_l", "LeftHandMiddle4": "middle_04_l",
    "RightHandMiddle1": "middle_01_r", "RightHandMiddle2": "middle_02_r", "RightHandMiddle3": "middle_03_r", "RightHandMiddle4": "middle_04_r",
    "LeftHandRing1": "ring_01_l", "LeftHandRing2": "ring_02_l", "LeftHandRing3": "ring_03_l", "LeftHandRing4": "ring_04_l",
    "RightHandRing1": "ring_01_r", "RightHandRing2": "ring_02_r", "RightHandRing3": "ring_03_r", "RightHandRing4": "ring_04_r",
    "LeftHandPinky1": "pinky_01_l", "LeftHandPinky2": "pinky_02_l", "LeftHandPinky3": "pinky_03_l", "LeftHandPinky4": "pinky_04_l",
    "RightHandPinky1": "pinky_01_r", "RightHandPinky2": "pinky_02_r", "RightHandPinky3": "pinky_03_r", "RightHandPinky4": "pinky_04_r",
}

def smpl22_skeleton(with_ends=False):
    names = list(SMPL22_NAMES)
    parents = list(SMPL22_PARENTS)
    offsets = [list(o) for o in SMPL22_OFFSETS_M]
    if with_ends:
        for nm, p, off in SMPL22_ENDS:
            names.append(nm)
            parents.append(p)
            offsets.append(off)
    return names, parents, offsets

# =====================================================================
# 3. BVH READER / WRITER
# =====================================================================
class BvhData:
    """Motion on an arbitrary BVH skeleton.
    local_pos (T,J,3): animated local position (== rest offset for static joints)
    local_quats (T,J,4): local rotations xyzw
    has_pos: which joints carry position channels
    """
    def __init__(self, names, parents, offsets_m, local_quats, local_pos, has_pos, fps):
        self.names = names
        self.parents = parents
        self.offsets_m = offsets_m
        self.local_quats = local_quats
        self.local_pos = local_pos
        self.has_pos = has_pos
        self.fps = fps

    @property
    def n_frames(self):
        return self.local_quats.shape[0]

    @property
    def root_index(self):
        return self.parents.index(-1)

    def world_fk(self):
        """Returns (world_pos (T,J,3), world_rot (T,J,3,3))."""
        T, J = self.local_quats.shape[:2]
        world_pos = np.zeros((T, J, 3), dtype=np.float64)
        world_rot = np.zeros((T, J, 3, 3), dtype=np.float64)
        ridx = self.root_index
        for t in range(T):
            world_pos[t, ridx] = self.local_pos[t, ridx]
            world_rot[t, ridx] = quat_to_mat(self.local_quats[t, ridx])
            for j in range(1, J):
                p = self.parents[j]
                if p < 0:
                    continue
                world_rot[t, j] = world_rot[t, p] @ quat_to_mat(self.local_quats[t, j])
                world_pos[t, j] = world_pos[t, p] + (world_rot[t, p] @ self.local_pos[t, j])
        return world_pos, world_rot

def flatten_motion(b: BvhData):
    """Normalize the root: if a static wrapper root has a child with position
    channels (e.g. Kimodo 'Root' wrapper -> 'Hips'), make that child the root.
    Returns a (possibly new) BvhData whose root carries the world motion."""
    ridx = b.root_index
    moved = ridx
    kids = [c for c in range(len(b.names)) if b.parents[c] == ridx]
    if b.has_pos[ridx] and b.n_frames > 1:
        rstd = float(np.std(b.local_pos[:, ridx, :]))
        for c in kids:
            if b.has_pos[c] and rstd <= 1e-6 and float(np.std(b.local_pos[:, c, :])) > 1e-6:
                moved = c
                break
    elif not b.has_pos[ridx]:
        for c in kids:
            if b.has_pos[c]:
                moved = c
                break
    if moved == ridx:
        return b
    keep = [moved]
    def add(k):
        for c in range(len(b.names)):
            if b.parents[c] == k:
                keep.append(c)
                add(c)
    add(moved)
    old2new = {old_j: new_j for new_j, old_j in enumerate(keep)}
    names = [b.names[j] for j in keep]
    offsets = [b.offsets_m[j] for j in keep]
    has_pos = [b.has_pos[j] for j in keep]
    parents = [old2new.get(b.parents[j], -1) for j in keep]
    local_quats = b.local_quats[:, keep, :]
    local_pos = b.local_pos[:, keep, :]
    return BvhData(names, parents, offsets, local_quats, local_pos, has_pos, b.fps)

def _euler_zyx_mat(rx, ry, rz):
    cx, sx = math.cos(rx), math.sin(rx)
    cy, sy = math.cos(ry), math.sin(ry)
    cz, sz = math.cos(rz), math.sin(rz)
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
    return Rz @ Ry @ Rx

def _quat_to_euler_zyx(q):
    R = quat_to_mat(q)
    sy = float(np.clip(-R[2, 0], -1.0, 1.0))
    ry = math.asin(sy)
    if abs(sy) < 0.99999:
        rx = math.atan2(R[2, 1], R[2, 2])
        rz = math.atan2(R[1, 0], R[0, 0])
    else:
        rx = math.atan2(-R[1, 2], R[1, 1])
        rz = 0.0
    return rx, ry, rz

def read_bvh(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = [ln.strip() for ln in f.read().splitlines()]
    names, parents, offsets, channels, has_pos = [], [], [], [], []
    stack = []
    in_motion = False
    i = 0
    fps, n_frames, data_rows = 30.0, 0, []
    n = len(lines)
    while i < n:
        s = lines[i]
        if s == "HIERARCHY":
            i += 1
            continue
        if s == "MOVTION":
            in_motion = True
            i += 1
            continue
        if not in_motion:
            if s.startswith("ROOT") or s.startswith("JOINT"):
                jname = s.split()[-1]
                idx = len(names)
                names.append(jname)
                parents.append(stack[-1] if stack else -1)
                offsets.append(None)
                channels.append(None)
                has_pos.append(False)
                stack.append(idx)
            elif s.startswith("OFFSET") and stack:
                if offsets[stack[-1]] is None:
                    offsets[stack[-1]] = [float(v) for v in s.split()[1:4]]
            elif s.startswith("CHANNELS") and stack:
                parts = s.split()
                chn = int(parts[1])
                ch = parts[2:2 + chn]
                channels[stack[-1]] = ch
                has_pos[stack[-1]] = "Xposition" in ch
            elif s == "End Site":
                # skip the end site block
                depth = 0
                started = False
                while i < n:
                    t = lines[i]
                    if t == "{":
                        depth += 1
                        started = True
                    elif t == "}":
                        depth -= 1
                        if started and depth == 0:
                            break
                    i += 1
            elif s == "}":
                if stack:
                    stack.pop()
        else:
            if s.startswith("Frames Per Second"):
                fps = float(s.split(":")[1])
            elif s.startswith("Number of Frames"):
                n_frames = int(s.split(":")[1])
            elif s and re.match(r"^[\d.\-eE+]+( [\d.\-eE+]+)+$", s):
                data_rows.append(s)
        i += 1
    for j in range(len(names)):
        if offsets[j] is None:
            offsets[j] = [0.0, 0.0, 0.0]
        if channels[j] is None:
            channels[j] = []
    data = np.array([[float(v) for v in r.split()] for r in data_rows[:n_frames]], dtype=np.float64)
    T = data.shape[0]
    J = len(names)
    local_quats = np.zeros((T, J, 4), dtype=np.float64)
    local_pos = np.array(offsets, dtype=np.float64).reshape(1, J, 3).repeat(T, axis=0)
    col_of = []
    col = 0
    for ch in channels:
        col_of.append(col)
        col += len(ch)
    for t in range(T):
        for jn in range(J):
            ch = channels[jn]
            if not ch:
                continue
            if "Xposition" in ch:
                c0 = col_of[jn]
                local_pos[t, jn] = (data[t, c0 + ch.index("Xposition")],
                                    data[t, c0 + ch.index("Yposition")],
                                    data[t, c0 + ch.index("Zposition")])
            if "Zrotation" in ch:
                c0 = col_of[jn]
                rz = math.radians(data[t, c0 + ch.index("Zrotation")])
                ry = math.radians(data[t, c0 + ch.index("Yrotation")])
                rx = math.radians(data[t, c0 + ch.index("Xrotation")])
                local_quats[t, jn] = mat_to_quat(_euler_zyx_mat(rx, ry, rz))
    return BvhData(names, parents, offsets, local_quats, local_pos, has_pos, fps)

def write_bvh(b: BvhData, path, unit_scale=1.0):
    out = ["HIERARCHY"]
    ridx = b.root_index
    def emit(j, depth):
        pad = "\t" * (depth + 1)
        kw = "ROOT" if j == ridx else "JOINT"
        out.append(f"{pad}{kw} {b.names[j]}")
        out.append(pad + "{")
        off = b.offsets_m[j]
        out.append(f"{pad}\tOFFSET {off[0]*unit_scale:.6f} {off[1]*unit_scale:.6f} {off[2]*unit_scale:.6f}")
        if b.has_pos[j]:
            out.append(f"{pad}\tCHANNELS 6 Xposition Yposition Zposition Zrotation Yrotation Xrotation")
        else:
            out.append(f"{pad}\tCHANNELS 3 Zrotation Yrotation Xrotation")
        kids = [k for k in range(len(b.names)) if b.parents[k] == j]
        for k in kids:
            emit(k, depth + 1)
        if not kids:
            out.append(f"{pad}\tEnd Site")
            out.append(f"{pad}\t\t{{")
            out.append(f"{pad}\t\t\tOFFSET 0.000000 0.000000 0.000000")
            out.append(f"{pad}\t\t}}")
        out.append(pad + "}")
    emit(ridx, 0)
    out.append("}  # HIERARCHY")
    out.append("")
    out.append("MOVTION")
    out.append(f"Frames Per Second: {b.fps:.4f}")
    out.append(f"Number of Frames: {b.n_frames}")
    for t in range(b.n_frames):
        vals = []
        for jn in range(len(b.names)):
            if b.has_pos[jn]:
                vals.extend(float(v) * unit_scale for v in b.local_pos[t, jn])
            rx, ry, rz = _quat_to_euler_zyx(b.local_quats[t, jn])
            vals.extend((math.degrees(rz), math.degrees(ry), math.degrees(rx)))
        out.append(" ".join(f"{v:.6f}" for v in vals))
    with open(path, "w") as f:
        f.write("\n".join(out) + "\n")

# =====================================================================
# 4. GLB (skeleton-only glTF 2.0 binary)
# =====================================================================
def _offsets_from_localpos(local_pos, J, T):
    """Static rest offsets: frame-0 local positions (root -> 0)."""
    out = []
    for j in range(J):
        out.append(np.asarray(local_pos[0, j, :], dtype=np.float64))
    return out

def write_glb_skeleton(names, parents, local_pos, local_quats, fps, path,
                       node_names=None, anim_name="motion"):
    J = len(names)
    T = int(local_quats.shape[0])
    if node_names is None:
        node_names = list(names)
    assert len(node_names) == J, "node_names must match skeleton"
    children = [[] for _ in range(J)]
    for j in range(J):
        if parents[j] >= 0:
            children[parents[j]].append(j)
    ridx = parents.index(-1)
    offsets_m = [np.zeros(3) if p < 0 else _off for p, _off in zip(parents, _offsets_from_localpos(local_pos, J, T))]
    dt = 1.0 / max(fps, 1e-6)

    buf = b""
    views, accs = [], []

    def f32view(vals):
        nonlocal buf
        vals = [float(v) for v in vals]
        assert len(buf) % 4 == 0
        data = struct.pack("<%df" % len(vals), *vals)
        views.append({"buffer": 0, "byteOffset": len(buf), "byteLength": len(data)})
        buf += data
        return len(views) - 1

    def accessor(view_idx, type_, count, mn, mx):
        accs.append({"bufferView": view_idx, "componentType": 5126, "count": count,
                     "type": type_, "min": [float(v) for v in mn], "max": [float(v) for v in mx]})
        return len(accs) - 1

    times = [t * dt for t in range(T)]
    a_time = accessor(f32view(times), "SCALAR", T, [times[0]], [times[-1]])

    rtx = np.asarray(local_pos[:, ridx, :], dtype=np.float32)
    a_rtrans = accessor(f32view(rtx.reshape(-1).tolist()), "VEC3", T, rtx.min(0).tolist(), rtx.max(0).tolist())

    qall = normalize_quats(np.asarray(local_quats, dtype=np.float64))
    rot_accs = []
    for j in range(J):
        q = qall[:, j, :]
        rot_accs.append(accessor(f32view(q.reshape(-1).tolist()), "VEC4", T,
                                 q.min(0).tolist(), q.max(0).tolist()))

    offs = np.asarray(local_pos[0, :, :], dtype=np.float32)
    a_offs = accessor(f32view(offs.reshape(-1).tolist()), "VEC3", J, offs.min(0).tolist(), offs.max(0).tolist())

    samplers = [{"input": a_time, "output": a_rtrans, "interpolation": "LINEAR"}]
    for j in range(J):
        samplers.append({"input": a_time, "output": rot_accs[j], "interpolation": "LINEAR"})
    channels = [{"sampler": 0, "target": {"node": ridx, "path": "translation"}}]
    for j in range(J):
        channels.append({"sampler": j + 1, "target": {"node": j, "path": "rotation"}})

    nodes = []
    for j in range(J):
        n = {"name": node_names[j]}
        n["translation"] = [0.0, 0.0, 0.0] if parents[j] < 0 else [float(v) for v in offsets_m[j]]
        n["rotation"] = [0.0, 0.0, 0.0, 1.0]
        if children[j]:
            n["children"] = children[j]
        nodes.append(n)

    gltf = {
        "asset": {"version": "2.0", "generator": "ai-mocap-colab"},
        "scene": 0,
        "scenes": [{"nodes": [ridx]}],
        "nodes": nodes,
        "animations": [{"name": anim_name, "samplers": samplers, "channels": channels}],
        "accessors": accs,
        "bufferViews": views,
        "buffers": [{"byteLength": len(buf)}],
    }
    js = json.dumps(gltf, separators=(",", ":")).encode("utf-8")
    js += b" " * ((4 - len(js) % 4) % 4)
    total = 12 + 8 + len(js) + 8 + len(buf)
    with open(path, "wb") as f:
        f.write(struct.pack("<4sII", b"glTF", 2, total))
        f.write(struct.pack("<II", len(js), 0x4E4F534A))
        f.write(js)
        f.write(struct.pack("<II", len(buf), 0x004E4942))
        f.write(buf)
    return path

# =====================================================================
# 5. ASCII FBX (skeleton + animation, UE5 friendly)
# =====================================================================
def write_fbx_skeleton(names, parents, local_pos, local_quats, fps, path,
                       node_names=None, anim_name="motion"):
    J = len(names)
    T = int(local_quats.shape[0])
    if node_names is None:
        node_names = list(names)
    ridx = parents.index(-1)
    fps_ms = 1000.0 / max(fps, 1e-6)
    n_curves = 3 + 4 * J
    offsets_m = [np.zeros(3) if p < 0 else _off for p, _off in zip(parents, _offsets_from_localpos(local_pos, J, T))]
    L = []
    A = L.append
    A("; FBX 7.4.0 project file")
    A("; ai-mocap-colab skeleton+animation export")
    A("")
    A("FBXHeaderExtension:  {")
    A("    FBXHeaderVersion: 1003")
    A("    FBXVersion: 7400")
    A('    Creator: "ai-mocap-colab"')
    A("    SceneInfo:  {")
    A('        Type: "Data"')
    A('        Version: 1000')
    A('        Properties70:  {')
    A('            P: "DocumentUrl", "string", "", "", ""')
    A(f'            P: "FPS", "double", "Number", "", "{fps:.4f}"')
    A('            P: "GlobalStart", "KTime", "Time", "", "0"')
    A('            P: "GlobalEnd", "KTime", "Time", "", "0"')
    A("        }")
    A("    }")
    A("}  # FBXHeaderExtension")
    A("")
    A("GlobalSettings:  {")
    A("    Version: 1000")
    A("    Properties70:  {")
    A('        P: "UpAxis", "int", "Integer", "", "1"')
    A('        P: "UpAxisSign", "int", "Integer", "", "1"')
    A('        P: "FrontAxis", "int", "Integer", "", "2"')
    A('        P: "FrontAxisSign", "int", "Integer", "", "1"')
    A('        P: "CoordAxis", "int", "Integer", "", "0"')
    A('        P: "CoordAxisSign", "int", "Integer", "", "1"')
    A('        P: "OriginalUpAxis", "int", "Integer", "", "1"')
    A('        P: "UnitScaleFactor", "double", "Number", "", "1"')
    A('        P: "OriginalUnitScaleFactor", "double", "Number", "", "1"')
    A("    }")
    A("}  # GlobalSettings")
    A("")
    A("Definitions:  {")
    A("    Version: 100")
    A(f"    Count: {2 + J + n_curves}")
    A('    ObjectType: "GlobalSettings"')
    A('    Object: "Model", "NULL", "Null"')
    for _ in range(J):
        A('    Object: "Model", "NULL", "Null"')
    A('    Object: "AnimationStack", "AnimStack", "AnimationStack"')
    A('    Object: "AnimationLayer", "AnimLayer", "AnimationLayer"')
    for _ in range(n_curves):
        A('    Object: "Curve", "AnimCurve", "Curve"')
    A("}  # Definitions")
    A("")
    A("Objects:  {")
    A(f"    Count: {1 + J + 1 + 1 + n_curves}")
    A('    Model: "Model::RootWrapper", "Null", "" {')
    A("        Version: 232")
    A("        Properties70:  {")
    A('            P: "Lcl Translation", "Lcl Translation", "", "A", 0, 0, 0')
    A('            P: "Lcl Rotation", "Lcl Rotation", "", "A", 0, 0, 0')
    A('            P: "Lcl Scaling", "Lcl Scaling", "", "A", 1, 1, 1')
    A('            P: "DefaultAttributeIndex", "int", "Integer", "", "0"')
    A("        }")
    A("        MultiLayer: 0")
    A("        MultiTake: 0")
    A('        TType: "Null"')
    A("    }  # Model")
    for j in range(J):
        off = offsets_m[j]
        A(f'    Model: "Model::{node_names[j]}", "Null", "" {{')
        A("        Version: 232")
        A("        Properties70:  {")
        A(f'            P: "Lcl Translation", "Lcl Translation", "", "A", {off[0]:.6f}, {off[1]:.6f}, {off[2]:.6f}')
        A('            P: "Lcl Rotation", "Lcl Rotation", "", "A", 0, 0, 0')
        A('            P: "Lcl Scaling", "Lcl Scaling", "", "A", 1, 1, 1')
        A('            P: "DefaultAttributeIndex", "int", "Integer", "", "0"')
        A("        }")
        A("        MultiLayer: 0")
        A("        MultiTake: 0")
        A('        TType: "Null"')
        A("    }  # Model")
    A(f'    AnimationStack: "AnimStack::{anim_name}", "AnimStack", "" {{')
    A('        Attributes: ""')
    A("    }  # AnimationStack")
    A(f'    AnimationLayer: "AnimLayer::{anim_name}", "AnimLayer", "" {{')
    A("    }  # AnimationLayer")

    def emit_curve(label, values_by_frame):
        n = len(values_by_frame)
        times = " ".join(str(int(round(t * fps_ms))) for t in range(n))
        vals = " ".join(f"{v:.8f}" for v in values_by_frame)
        A(f'    Curve: "AnimCurve::{label}", "AnimCurve", "" {{')
        A("        Dimensions: 1")
        A(f"        KeyTime: *{n} {{ {times} }}")
        A(f"        KeyValueFloat: *{n} {{ {vals} }}")
        A("    }  # Curve")

    for c in range(3):
        emit_curve(f"tr_{c}", [float(local_pos[t, ridx, c]) for t in range(T)])
    for j in range(J):
        for c in range(4):
            emit_curve(f"node{j}_rot_{c}", [float(local_quats[t, j, c]) for t in range(T)])
    A("}  # Objects")
    A("")
    A("Connections:  {")
    for j in range(J):
        if parents[j] >= 0:
            A(f"    C: \"OO\", {parents[j] + 1}, {j + 1}, \"\"")
        else:
            A(f"    C: \"OO\", 0, {j + 1}, \"\"")
    stack_id = 1 + J
    layer_id = stack_id + 1
    first_curve = layer_id + 1
    A(f"    C: \"OO\", {stack_id}, {layer_id}, \"\"")
    for j in range(J):
        A(f"    C: \"OO\", {layer_id}, {j + 1}, \"\"")
    ci = 0
    for j in range(J):
        model_id = j + 1
        if j == ridx:
            for c in range(3):
                A(f"    C: \"OO\", {model_id}, {first_curve + ci}, \"\"")
                ci += 1
        for c in range(4):
            A(f"    C: \"OO\", {model_id}, {first_curve + ci}, \"\"")
            ci += 1
    A("}  # Connections")
    with open(path, "w") as f:
        f.write("\n".join(L) + "\n")
    return path

# =====================================================================
# 6. FK + MODEL-OUTPUT CONVERSIONS
# =====================================================================
def fk_world_positions(local_quats, local_pos, parents):
    T, J, _ = local_quats.shape
    world_pos = np.zeros((T, J, 3), dtype=np.float64)
    world_rot = np.zeros((T, J, 3, 3), dtype=np.float64)
    ridx = parents.index(-1)
    for t in range(T):
        world_pos[t, ridx] = local_pos[t, ridx]
        world_rot[t, ridx] = quat_to_mat(local_quats[t, ridx])
        for j in range(1, J):
            p = parents[j]
            if p < 0:
                continue
            world_rot[t, j] = world_rot[t, p] @ quat_to_mat(local_quats[t, j])
            world_pos[t, j] = world_pos[t, p] + (world_rot[t, p] @ local_pos[t, j])
    return world_pos

def gts_to_motion(gts, root_mode="floor", target_pelvis_height=0.95, fps=20.0):
    """MoMask/MLD gts (T,263) -> BvhData on the SMPL-22 skeleton.
    Layout: [yaw_vel1 | vel_xz2 | root_y1 | ric_pos63 | rot6d(21 joints)126 | local_vel66 | foot4]
    Body rotations are cont6d (first two columns of each local rotation matrix).
    """
    gts = np.asarray(gts, dtype=np.float64)
    T = gts.shape[0]
    # root: pure Y-axis rotation from cumulative yaw velocity (MLD stores half-angles)
    half_ang = np.zeros(T)
    if T > 1:
        half_ang[1:] = np.cumsum(gts[:T - 1, 0])
    # recover_root_rot_pos: r_pos[1:, (0,2)] = qrot(qinv(rq), [vx, 0, vz]); cumsum; y = root_y
    r_q = np.zeros((T, 4))  # library convention (x, y, z, w)
    r_q[:, 1] = np.sin(half_ang)
    r_q[:, 3] = np.cos(half_ang)
    vel = np.zeros((T, 3))
    if T > 1:
        vel[1:, 0] = gts[:-1, 1]
        vel[1:, 2] = gts[:-1, 2]
    r_pos = np.zeros((T, 3))
    if T > 1:
        rv = quat_to_mat(r_q)
        rvel_world = np.einsum("tij,tj->ti", np.linalg.inv(rv), vel)
        r_pos[1:] = np.cumsum(rvel_world[1:], axis=0)
    r_pos[:, 1] = gts[:, 3]
    # body: cont6d -> local rotation matrices -> quats
    local_quats = np.zeros((T, 22, 4), dtype=np.float64)
    local_quats[:, 0] = r_q
    rot6 = gts[:, 67:193]
    for j in range(21):
        c1 = rot6[:, j * 6:j * 6 + 3]
        c2 = rot6[:, j * 6 + 3:j * 6 + 6]
        c3 = np.cross(c1, c2)
        c1n = c1 / np.maximum(np.linalg.norm(c1, axis=1, keepdims=True), 1e-12)
        c2n = c2 - np.einsum("ij,ij->i", c2, c1n)[:, None] * c1n
        c2n = c2n / np.maximum(np.linalg.norm(c2n, axis=1, keepdims=True), 1e-12)
        R = np.stack([c1n, c2n, np.cross(c1n, c2n)], axis=2)
        local_quats[:, j + 1] = mat_to_quat(R)
    local_quats = normalize_quats(local_quats)
    names, parents, offsets = smpl22_skeleton(with_ends=False)
    local_pos = np.array(offsets, dtype=np.float64).reshape(1, 22, 3).repeat(T, axis=0)
    if root_mode == "floor":
        ry = r_pos[:, 1]
        local_pos[:, 0, 1] = target_pelvis_height + (ry - ry[0])
        local_pos[:, 0, 0] = r_pos[:, 0]
        local_pos[:, 0, 2] = r_pos[:, 2]
    elif root_mode == "inplace":
        local_pos[:, 0, :] = np.stack([np.zeros(T), np.full(T, target_pelvis_height), np.zeros(T)], axis=1)
    else:  # origin: frame 0 como referência
        local_pos[:, 0, :] = r_pos - r_pos[0]
    has_pos = [False] * 22
    has_pos[0] = True
    return BvhData(names, parents, offsets, local_quats, local_pos, has_pos, fps)

def poses_to_motion(Rh, poses, trans, target_pelvis_height=0.95, fps=30.0, root_mode="floor"):
    """HY-Motion smpl_data: Rh (T,3,3)|(T,9), poses (T, >=63*3) aa, trans (T,3).
    Body = first 22 joints (root + 21)."""
    Rh = np.asarray(Rh, dtype=np.float64)
    root_quat = mat_to_quat(Rh.reshape(-1, 3, 3))
    T = root_quat.shape[0]
    poses = np.asarray(poses, dtype=np.float64)
    body_aa = poses.reshape(T, -1, 3)[:, :21, :]
    trans = np.asarray(trans, dtype=np.float64).reshape(T, 3)
    local_quats = np.zeros((T, 22, 4), dtype=np.float64)
    local_quats[:, 0] = root_quat
    local_quats[:, 1:] = mat_to_quat(aa_to_mat(body_aa))
    local_quats = normalize_quats(local_quats)
    names, parents, offsets = smpl22_skeleton(with_ends=False)
    local_pos = np.array(offsets, dtype=np.float64).reshape(1, 22, 3).repeat(T, axis=0)
    if root_mode == "floor":
        local_pos[:, 0] = np.stack([trans[:, 0], target_pelvis_height + (trans[:, 1] - trans[0, 1]), trans[:, 2]], axis=1)
    elif root_mode == "inplace":
        local_pos[:, 0] = np.stack([np.zeros(T), np.full(T, target_pelvis_height), np.zeros(T)], axis=1)
    else:  # origin: frame 0 como referência
        local_pos[:, 0] = trans - trans[0]
    has_pos = [False] * 22
    has_pos[0] = True
    return BvhData(names, parents, offsets, local_quats, local_pos, has_pos, fps)

def with_ue_names(b: BvhData, mapping):
    return [mapping.get(n, n) for n in b.names]

# =====================================================================
# 7. PREVIEW HTML (three.js stick figure)
# =====================================================================
def build_preview_html(samples, title="AI Motion Preview"):
    data = []
    for s in samples:
        P = np.asarray(s["positions"]).reshape(-1, 3)
        data.append({
            "label": s["label"],
            "parents": s["parents"],
            "T": len(P),
            "fps": s.get("fps", 30.0),
            "pos": [[round(float(v), 4) for v in row] for row in P],
        })
    payload = json.dumps(data)
    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>{title}</title>
<style>body{{margin:0;background:#101018;font-family:sans-serif;color:#eee}}
#hud{{position:fixed;top:10px;left:10px;z-index:10;background:#1c1c28cc;padding:10px 14px;border-radius:10px;font-size:13px;line-height:1.6}}
#ctrl{{position:fixed;bottom:10px;left:50%;transform:translateX(-50%);z-index:10;background:#1c1c28cc;padding:10px 14px;border-radius:10px;display:flex;gap:10px;align-items:center}}
button{{background:#3a3a55;color:#eee;border:0;border-radius:6px;padding:6px 12px;cursor:pointer}}
input[type=range]{{width:260px}}</style></head><body>
<div id="hud"><b>{title}</b><br><span id="lbl"></span></div>
<div id="ctrl">
 <button id="play">&#9208; Pause</button>
 <input id="scrub" type="range" min="0" max="100" value="0">
 <span id="tinfo"></span>
 <select id="sample"></select>
</div>
<script type="importmap">{{"imports":{{"three":"https://unpkg.com/three@0.160.0/build/three.module.js","three/addons/":"https://unpkg.com/three@0.160.0/examples/jsm/"}}}}</script>
<script type="module">
import * as THREE from 'three';
import {{OrbitControls}} from 'three/addons/controls/OrbitControls.js';
const DATA = {payload};
const scene = new THREE.Scene();
scene.background = new THREE.Color(0x101018);
const cam = new THREE.PerspectiveCamera(50, innerWidth/innerHeight, 0.01, 100);
cam.position.set(1.6, 1.4, 2.4);
const renderer = new THREE.WebGLRenderer({{antialias:true}});
renderer.setSize(innerWidth, innerHeight);
document.body.appendChild(renderer.domElement);
const controls = new OrbitControls(cam, renderer.domElement);
controls.target.set(0, 0.9, 0);
scene.add(new THREE.GridHelper(6, 12, 0x444466, 0x2a2a3a));
scene.add(new THREE.AmbientLight(0xffffff, 0.9));
const dir = new THREE.DirectionalLight(0xffffff, 1.2); dir.position.set(2,4,3); scene.add(dir);
let cur = 0, frame = 0, playing = true;
const sel = document.getElementById('sample');
DATA.forEach((d,i)=>{{ const o=document.createElement('option'); o.value=i; o.textContent=d.label; sel.appendChild(o); }});
sel.onchange = ()=>{{ cur=+sel.value; frame=0; rebuild(); syncUI(); }};
function rebuild(){{
  const old = scene.getObjectByName('fig'); if (old) scene.remove(old);
  const d = DATA[cur];
  const g = new THREE.Group(); g.name='fig';
  const mat = new THREE.LineBasicMaterial({{color:0x66ccff}});
  const segs = [];
  for (let j=1;j<d.parents.length;j++) if (d.parents[j]>=0) segs.push([d.parents[j], j]);
  const lines = [];
  for (const [a,b] of segs){{
    const geo = new THREE.BufferGeometry();
    geo.setAttribute('position', new THREE.BufferAttribute(new Float32Array(6),3));
    const line = new THREE.Line(geo, mat); g.add(line); lines.push(line);
  }}
  const sphereGeo = new THREE.SphereGeometry(0.032, 12, 12);
  const joints = [];
  for (let j=0;j<d.parents.length;j++){{
    const m = new THREE.MeshStandardMaterial({{color: j===0?0xffcc44:0x66ff99}});
    const s = new THREE.Mesh(sphereGeo, m); g.add(s); joints.push(s);
  }}
  g._lines = lines; g._joints = joints; g._segs = segs;
  scene.add(g);
}}
function setFrame(f){{
  const d = DATA[cur];
  frame = Math.max(0, Math.min(d.T-1, f|0));
  const g = scene.getObjectByName('fig'); if(!g) return;
  const off = frame*d.pos.length;
  for (let j=0;j<d.parents.length;j++){{
    const p = d.pos[off + j*3];
    g._joints[j].position.set(p[0], p[1], p[2]);
  }}
  g._lines.forEach((ln,i)=>{{
    const a = d.pos[off + g._segs[i][0]*3], b = d.pos[off + g._segs[i][1]*3];
    const arr = ln.geometry.attributes.position;
    arr.setXYZ(0, a[0],a[1],a[2]); arr.setXYZ(1, b[0],b[1],b[2]);
    arr.needsUpdate = true;
  }});
  document.getElementById('scrub').value = frame;
  document.getElementById('tinfo').textContent = (frame+1)+' / '+d.T+'  ('+(frame/d.fps).toFixed(1)+'s)';
  document.getElementById('lbl').textContent = d.label;
}}
rebuild(); syncUI();
function syncUI(){{ setFrame(frame); }}
let last = performance.now();
function tick(now){{
  requestAnimationFrame(tick);
  const d = DATA[cur];
  if (playing){{
    frame += (now-last)/1000*d.fps;
    if (frame >= d.T) frame = 0;
    setFrame(frame);
  }}
  last = now;
  renderer.render(scene, cam);
}}
requestAnimationFrame(tick);
document.getElementById('play').onclick = ()=>{{ playing=!playing; document.getElementById('play').textContent = playing?'\\u23F8 Pause':'\\u25B6 Play'; }};
document.getElementById('scrub').oninput = (e)=>{{ playing=false; document.getElementById('play').textContent='\\u25B6 Play'; setFrame(+e.target.value); }};
addEventListener('resize', ()=>{{ cam.aspect=innerWidth/innerHeight; cam.updateProjectionMatrix(); renderer.setSize(innerWidth, innerHeight); }});
</script></body></html>"""


In [ ]:
#@title ③ SETUP — instala o modelo da sessão (rode 1x)
# ════════════════════════ ③ SETUP (ambiente + pesos) ════════════════════════
# Rode 1x por sessao. Usa cache em /content (e no Drive, se marcado no CONFIG).
import os, sys, shutil, subprocess, glob, time
import torch

if not CFG.get("_ok"):
    raise RuntimeError("Rode a celula ① CONFIG primeiro.")

MODEL = CFG["model"]
CACHE = "/content/ai_mocap_cache"
REPO  = "/content/ai_mocap"
LOG   = f"{CACHE}/setup.log"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(REPO, exist_ok=True)
open(LOG, "a").close()

DRIVE = os.path.expanduser("~/drive/MyDrive/ai_mocap_cache")
USE_DRIVE = CFG.get("use_drive", False) and os.path.isdir(os.path.expanduser("~/drive/MyDrive"))
if CFG.get("use_drive") and not USE_DRIVE:
    print("ATT: Google Drive nao montado -> cache apenas local.")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (vai demorar!)")


def run(cmd, timeout=None, tail=60):
    'Roda um comando com saida AO VIVO e, se falhar, mostra o fim do log.'
    print("$ " + cmd, flush=True)
    p = subprocess.run(["bash", "-o", "pipefail", "-c", cmd + f" 2>&1 | tee -a {LOG}"],
                       timeout=timeout)
    if p.returncode != 0:
        try:
            txt = open(LOG, errors="replace").read().splitlines()[-tail:]
            print("-" * 70)
            print("\n".join(txt))
            print("-" * 70)
        except Exception:
            pass
        raise RuntimeError(f"FALHOU: {cmd}\n(log completo em {LOG})")
    return True


def pip(specs, label=""):
    'pip install tolerante: tenta o bloco inteiro e, se falhar, um por um.'
    q = " ".join('"%s"' % sp for sp in specs)
    try:
        return run("pip install -q " + q)
    except Exception:
        print("  (bloco '%s' falhou -> tentando 1 a 1)" % (label or q))
        ok = True
        for sp in specs:
            try:
                run('pip install -q "%s"' % sp)
            except Exception as e:
                print("  ATT: nao consegui instalar", sp, "->", str(e)[:160])
                ok = False
        return ok


def can_import(mod):
    'True se o modulo importa num processo novo (testado fora da pasta do repo).'
    kw = {"cwd": "/content"} if os.path.isdir("/content") else {}
    try:
        r = subprocess.run([sys.executable, "-c", "import %s" % mod],
                           capture_output=True, text=True, **kw)
        return r.returncode == 0
    except Exception:
        return False


def get_repo(name, git_url):
    local = f"{REPO}/{name}"
    if os.path.isdir(local) and os.listdir(local):
        print(f"  OK repositorio em cache: {local}")
        return local
    d = f"{DRIVE}/{name}" if USE_DRIVE else None
    if d and os.path.isdir(d) and os.listdir(d):
        print(f"  <- Do Drive: {d}"); shutil.copytree(d, local)
    else:
        print(f"  <- Baixando {name}...")
        run(f"git clone -q --depth 1 {git_url} {local}")
        if USE_DRIVE:
            os.makedirs(DRIVE, exist_ok=True); shutil.copytree(local, d, dirs_exist_ok=True)
    return local


def get_weights(name, test, download_cmd, drive_sub=None):
    local = f"{CACHE}/{name}"
    if os.path.isdir(local) and test(local):
        print(f"  OK pesos em cache: {local}")
        return local
    d = f"{DRIVE}/{name}" if USE_DRIVE else None
    if d and os.path.isdir(d) and test(d):
        print(f"  <- Pesos do Drive: {d}"); shutil.copytree(d, local)
    else:
        print(f"  <- Baixando pesos de {name} (pode demorar)...")
        run(download_cmd.format(CACHE=CACHE, name=name))
        if USE_DRIVE:
            os.makedirs(DRIVE, exist_ok=True); shutil.copytree(local, f"{DRIVE}/{name}", dirs_exist_ok=True)
    return local


def mem_info():
    ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
    return vram, ram


def hf_token():
    'Pega o token do HF: Secrets do Colab (HF_TOKEN) ou variavel de ambiente.'
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            tok = None
    return tok


# dependencias do Kimodo em blocos (o resolvedor do pip trabalha melhor assim)
KIMODO_CORE = ["transformers==5.1.0", "peft>=0.18", "accelerate>=0.30",
               "safetensors", "huggingface_hub", "tokenizers"]
KIMODO_CFG  = ["hydra-core>=1.3", "omegaconf>=2.3", "einops>=0.7", "tqdm>=4.0",
               "packaging>=21.0", "pydantic>=2.0", "filelock>=3.20.3"]
KIMODO_MISC = ["trimesh>=3.21.7", "pillow>=9.0", "bvhio", "scipy>=1.10"]

VRAM, RAM = mem_info()
print(f"  VRAM: {VRAM:.1f} GB | RAM: {RAM:.1f} GB")
HF_TOK = hf_token()
os.environ["HF_HOME"] = f"{CACHE}/hf"
os.environ["HUGGINGFACE_CACHE_DIR"] = f"{CACHE}/hf/hub"
os.makedirs(f"{CACHE}/hf", exist_ok=True)

t0 = time.time()
if MODEL == "kimodo":
    # --- token do Hugging Face: o text encoder do Kimodo usa o Llama-3-8B (gated)
    if HF_TOK:
        os.environ["HF_TOKEN"] = HF_TOK
        os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOK
        try:
            from huggingface_hub import login, HfApi
            login(token=HF_TOK, add_to_git_credential=False)
            print("  HF: logado como", HfApi(token=HF_TOK).whoami().get("name", "?"))
        except Exception as e:
            print("  ATT: nao consegui validar o token HF ->", str(e)[:160])
    else:
        print("  ATT: HF_TOKEN nao encontrado.")
        print("       Colab: icone de chave (esquerda) -> Secrets -> nome: HF_TOKEN")
        print("       Sem ele o Kimodo NAO baixa o Llama-3-8B do encoder de texto.")

    repo = get_repo("kimodo", "https://github.com/nv-tlabs/kimodo")
    SKIP = "SKIP_MOTION_CORRECTION_IN_SETUP=1"   # nao compila o C++ do pos-processamento
    ok = False

    if CFG.get("compile_postprocess"):
        print("  Compilando o pos-processamento em C++ (leva alguns minutos)...")
        try:
            run("apt-get -qq update && apt-get -qq install -y cmake libeigen3-dev "
                "pybind11-dev || pip install -q cmake")
            run(f"pip install -q {repo}")
            ok = can_import("kimodo")
        except Exception as e:
            print("  ATT: instalação completa falhou ->", str(e)[:160])

    if not ok:
        print("  Instalando o Kimodo (sem o C++ do pos-processamento: nao precisa de cmake)...")
        try:
            run(f"{SKIP} pip install -q --no-deps {repo}")
            pip(KIMODO_CORE, "core")
            pip(KIMODO_CFG, "config")
            pip(KIMODO_MISC, "misc")
            ok = can_import("kimodo")
        except Exception as e:
            print("  ATT:", str(e)[:160])

    if not ok:
        print("  2a tentativa: deixando o pip resolver as dependencias do proprio Kimodo...")
        try:
            run(f"{SKIP} pip install -q {repo}")
            ok = can_import("kimodo")
        except Exception as e:
            print("  ATT:", str(e)[:160])

    if not ok:
        print("  3a tentativa: instalar cmake/eigen e compilar tudo...")
        run("apt-get -qq update && apt-get -qq install -y cmake libeigen3-dev "
            "pybind11-dev || pip install -q cmake")
        run(f"pip install -q {repo}")
        ok = can_import("kimodo")

    if not ok:
        raise RuntimeError("Nao consegui importar o Kimodo. Log completo em " + LOG)

    has_mc = can_import("motion_correction")
    enc = CFG.get("text_encoder", "auto")
    if enc == "auto":
        enc = "gpu" if VRAM >= 20 else ("cpu" if RAM >= 24 else "4bit")
    if enc in ("gpu", "4bit") and not torch.cuda.is_available():
        print("  ATT: sem GPU -> encoder no CPU")
        enc = "cpu"
    if enc == "cpu" and RAM < 20:
        print("  ATT: encoder no CPU com so %.1f GB de RAM - provavel OOM." % RAM)
        print("       No CONFIG escolha 'Encoder de texto' = 4bit.")
    if enc == "4bit":
        pip(["bitsandbytes"], "bitsandbytes")
    STATE.update(model=MODEL, weights_ready=True, text_encoder=enc,
                 has_motion_correction=has_mc)
    print("  Kimodo instalado.")
    print("  Encoder de texto:", enc, "->", {"4bit": "GPU 4-bit", "gpu": "GPU bf16",
                                             "cpu": "CPU bf16"}[enc])
    print("  Pos-processamento (foot-skate):", "disponivel" if has_mc else "indisponivel (gera com --no-postprocess)")

elif MODEL == "hymotion":
    repo = get_repo("hymotion", "https://github.com/Tencent-Hunyuan/HY-Motion-1.0")
    print("  Instalando dependencias do HY-Motion...")
    run(f"pip install -q -r {repo}/requirements.txt || echo 'alguns pacotes falharam (ok se torch ja existe)'")
    get_weights(
        "hymotion_ckpt",
        test=lambda p: os.path.exists(f"{p}/HY-Motion-1.0-Lite/config.yml"),
        download_cmd=(
            "huggingface-cli download tencent/HY-Motion-1.0 --include 'HY-Motion-1.0-Lite/*' "
            "--local-dir {CACHE}/{name}"
        ),
    )
    STATE.update(model=MODEL, weights_ready=True)

elif MODEL == "momask":
    repo = get_repo("momask", "https://github.com/EricGuo5513/momask-codes")
    print("  Instalando CLIP (encoder de texto do MoMask)...")
    run("pip install -q git+https://github.com/openai/CLIP.git")
    get_weights(
        "momask_ckpt",
        test=lambda p: len(glob.glob(f"{p}/**/opt.txt", recursive=True)) > 0,
        download_cmd=(
            "gdown -q --fuzzy 'https://drive.google.com/file/d/1vXS7SHJBgWPt59wupQ5UUzhFObrnGkQ0/view' "
            "-O {CACHE}/momask_dl.zip && mkdir -p {CACHE}/{name} && "
            "unzip -q {CACHE}/momask_dl.zip -d {CACHE}/{name} && rm {CACHE}/momask_dl.zip"
        ),
    )
    # o MoMask espera: <repo>/checkpoints/t2m/<dataset_name>/... -> faz o link
    ck = f"{CACHE}/momask_ckpt"
    opts = glob.glob(f"{ck}/**/opt.txt", recursive=True)
    if opts:
        sys.path.insert(0, repo)
        from options.eval_option import EvalT2MOptions
        _opt = EvalT2MOptions().parse()
        vq_dir = os.path.dirname(opts[0])              # .../<dataset>/<model>
        ds_dir = os.path.dirname(vq_dir)               # .../<dataset>
        link = f"{repo}/checkpoints/t2m/{_opt.dataset_name}"
        os.makedirs(os.path.dirname(link), exist_ok=True)
        if os.path.islink(link):
            os.remove(link)
        elif os.path.exists(link):
            raise RuntimeError(f"{link} ja existe - remova manualmente")
        os.symlink(ds_dir, link)
        print(f"  OK checkpoints linkados: {link} -> {ds_dir}")
    STATE.update(model=MODEL, weights_ready=True)

print(f"SETUP concluido em {time.time()-t0:.0f}s. Modelo: {MODEL}")
print("Agora rode a celula ④ GENERATE.")

In [ ]:
#@title ④ GENERATE — gerar as animações
# ════════════════════════ ④ GENERATE ════════════════════════
# Gera as variacoes da animacao com o modelo da sessao.
import os, sys, glob, subprocess, datetime, shlex

if not CFG.get("_ok"):
    raise RuntimeError("Rode a celula ① CONFIG primeiro.")
if STATE["model"] != CFG["model"]:
    if STATE["loaded"]:
        raise RuntimeError(
            f"Esta sessao esta usando '{STATE['model']}'. Para trocar de modelo: "
            "Runtime -> Restart runtime (1 modelo por sessao, de proposito).")
    raise RuntimeError("Rode a celula ③ SETUP primeiro.")

if STATE["loaded"]:
    print(f"  {CFG['model']} ja configurado nesta sessao -> gerando (sem baixar nada)")
else:
    if not STATE["weights_ready"]:
        raise RuntimeError("Rode a celula ③ SETUP primeiro.")
    print(f"  {CFG['model']}: 1a geracao desta sessao (o encoder recarrega do cache local)")

KIMODO_RUNNER = r'''# -*- coding: utf-8 -*-
# Chama o CLI do Kimodo aplicando (se preciso) o ajuste do encoder de texto.
# Roda como subprocesso: se estourar a memoria, o kernel do Colab sobrevive.
import os, sys

mode = os.environ.get("KM_ENCODER", "cpu")

if mode == "4bit":
    import torch
    from transformers import BitsAndBytesConfig
    from kimodo.model.llm2vec.llm2vec import LLM2Vec

    _orig_from_pretrained = LLM2Vec.from_pretrained.__func__

    def _from_pretrained_4bit(cls, *args, **kwargs):
        kwargs.setdefault("quantization_config", BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        ))
        kwargs.setdefault("device_map", {"": 0})
        print("[kimodo] text encoder: Llama-3-8B em 4-bit (NF4) na GPU", flush=True)
        return _orig_from_pretrained(cls, *args, **kwargs)

    LLM2Vec.from_pretrained = classmethod(_from_pretrained_4bit)

    # rede de seguranca: alguns builds reclamam ao mover um modelo quantizado
    _orig_to = LLM2Vec.to

    def _to(self, device, *a, **k):
        try:
            return _orig_to(self, device, *a, **k)
        except Exception as e:
            print("[kimodo] ignorando .to(%s): %s" % (device, e), flush=True)
            return self

    LLM2Vec.to = _to

sys.argv = ["kimodo.scripts.generate"] + sys.argv[1:]
from kimodo.scripts.generate import main
main()
'''


ts = datetime.datetime.now().strftime("%H%M%S")
OUT = f"/content/ai_mocap_out/{ts}"
os.makedirs(OUT, exist_ok=True)
STATE["motions"] = []
MODEL = CFG["model"]

if MODEL == "kimodo":
    enc = STATE.get("text_encoder", "cpu")
    RUNNER = f"{CACHE}/kimodo_run.py"
    with open(RUNNER, "w", encoding="utf-8") as f:
        f.write(KIMODO_RUNNER)

    env = os.environ.copy()
    env.update({
        "HF_HOME": f"{CACHE}/hf",
        "HF_HUB_CACHE": f"{CACHE}/hf/hub",
        "HUGGINGFACE_CACHE_DIR": f"{CACHE}/hf/hub",
        "TRANSFORMERS_CACHE": f"{CACHE}/hf/transformers",
        "TEXT_ENCODER_MODE": "local",                 # nao procura o servico de encoder
        "TEXT_ENCODER_DEVICE": "cuda" if enc in ("gpu", "4bit") else "cpu",
        "KM_ENCODER": enc,
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "TOKENIZERS_PARALLELISM": "false",
    })
    tok = globals().get("HF_TOK") or os.environ.get("HF_TOKEN")
    if tok:
        env["HF_TOKEN"] = tok
        env["HUGGINGFACE_HUB_TOKEN"] = tok

    stem = f"{OUT}/gen"
    args = [sys.executable, RUNNER, CFG["prompt"],
            "--model", "Kimodo-SOMA-RP-v1.1",
            "--duration", str(CFG["duration"]),
            "--num_samples", str(CFG["samples"]),
            "--seed", str(CFG["seed"]),
            "--bvh", "--bvh_standard_tpose",
            "--output", stem]
    if not STATE.get("has_motion_correction"):
        args.append("--no-postprocess")   # o C++ do pos-processamento nao foi compilado
    cmd = " ".join(shlex.quote(a) for a in args)
    print("$ " + cmd, flush=True)

    genlog = f"{OUT}/generate.log"
    p = subprocess.run(["bash", "-o", "pipefail", "-c", cmd + f" 2>&1 | tee {genlog}"],
                       env=env, cwd="/content")
    if p.returncode != 0:
        txt = ""
        try:
            txt = open(genlog, errors="replace").read().lower()
        except Exception:
            pass
        if "out of memory" in txt or "cuda oom" in txt or "killed" in txt:
            print("  DICA: faltou memoria. No CONFIG tente 'Encoder de texto' = 4bit,")
            print("        menos variações / duração menor, ou um runtime com mais VRAM.")
        if ("401" in txt or "403" in txt or "gated" in txt
                or "restricted" in txt or "not authorized" in txt):
            print("  DICA: sem acesso ao modelo do encoder de texto (Llama-3-8B).")
            print("        Aceite a licenca em https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct")
            print("        e confira o Secret HF_TOKEN do Colab.")
        raise RuntimeError(f"Falha na geracao do Kimodo (log: {genlog})")

    STATE["motions"] = [{"model": "kimodo", "bvh": bp}
                        for bp in sorted(glob.glob(stem + "*.bvh") + glob.glob(stem + "/*.bvh"))]
    print(f"  {len(STATE['motions'])} animacao(oes) gerada(s) -> {OUT}")

elif MODEL == "hymotion":
    sys.path.insert(0, f"{REPO}/hymotion")
    if STATE.get("hymotion_rt") is None:
        from hymotion.utils.t2m_runtime import T2MRuntime
        ck = f"{CACHE}/hymotion_ckpt/HY-Motion-1.0-Lite"
        STATE["hymotion_rt"] = T2MRuntime(
            config_path=f"{ck}/config.yml",
            ckpt_name=f"{ck}/latest.ckpt",
            disable_prompt_engineering=True,   # sem LLM de reescrita/duracao (nao precisa)
        )
        print("  HY-Motion carregado (fica em memoria pelo resto da sessao)")
    seeds = ",".join(str(CFG["seed"] + i) for i in range(CFG["samples"]))
    base = f"{OUT}/gen"
    html, _, _ = STATE["hymotion_rt"].generate_motion(
        text=CFG["prompt"],
        seeds_csv=seeds,
        duration=float(CFG["duration"]),
        cfg_scale=5.0,
        output_format="dict",
        output_dir=OUT,
        output_filename=base,
    )
    STATE["hymotion_html"] = html
    STATE["motions"] = [{"model": "hymotion", "npz": p} for p in sorted(glob.glob(f"{base}_*.npz"))]
    print(f"  {len(STATE['motions'])} animacao(oes) gerada(s) -> {OUT}")

elif MODEL == "momask":
    os.chdir(f"{REPO}/momask")
    sys.path.insert(0, f"{REPO}/momask")
    if STATE.get("momask") is None:
        import numpy as np, torch, torch.nn.functional as F
        from utils.fixseed import fixseed
        from options.eval_option import EvalT2MOptions
        from utils.get_opt import get_opt
        from gen_t2m import load_vq_model, load_res_model, load_trans_model, load_len_estimator

        parser = EvalT2MOptions()
        opt = parser.parse()
        fixseed(CFG["seed"])
        opt.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        dim_pose = 263
        root_dir = os.path.join(opt.checkpoints_dir, opt.dataset_name)
        model_opt = get_opt(os.path.join(root_dir, opt.name, "opt.txt"), device=opt.device)
        vq_opt = get_opt(os.path.join(root_dir, model_opt.vq_name, "opt.txt"), device=opt.device)
        vq_opt.dim_pose = dim_pose
        vq_model, vq_opt = load_vq_model(vq_opt)
        model_opt.num_tokens = vq_opt.nb_code
        model_opt.num_quantizers = vq_opt.num_quantizers
        model_opt.code_dim = vq_opt.code_dim
        res_opt = get_opt(os.path.join(root_dir, model_opt.res_name, "opt.txt"), device=opt.device)
        res_model = load_res_model(res_opt, vq_opt, opt)
        t2m = load_trans_model(model_opt, opt, "latest.tar")
        length_estimator = load_len_estimator(opt)
        for m_ in (t2m, vq_model, res_model, length_estimator):
            m_.eval().to(opt.device)
        mean = np.load(os.path.join(root_dir, model_opt.vq_name, "meta", "mean.npy"))
        std = np.load(os.path.join(root_dir, model_opt.vq_name, "meta", "std.npy"))
        frames = max(8, (int(CFG["duration"]) * 20) // 4 * 4)   # 20 fps, multiplo de 4
        STATE["momask"] = dict(opt=opt, t2m=t2m, vq=vq_model, res=res_model,
                               mean=mean, std=std, frames=frames)
        print(f"  MoMask carregado ({frames} frames @20fps)")
    st = STATE["momask"]
    import numpy as np, torch
    token_lens = torch.tensor([st["frames"] // 4]).to(st["opt"].device)
    prompt_list = [CFG["prompt"]]
    for i in range(CFG["samples"]):
        from utils.fixseed import fixseed
        fixseed(CFG["seed"] + i)
        mids = st["t2m"].generate(prompt_list, token_lens, timesteps=st["opt"].time_steps,
                                  cond_scale=st["opt"].cond_scale,
                                  temperature=st["opt"].temperature,
                                  topk_filter_thres=st["opt"].topkr,
                                  gsample=st["opt"].gumbel_sample)
        mids = st["res"].generate(mids, prompt_list, token_lens, temperature=1, cond_scale=5)
        pred = st["vq"].forward_decoder(mids).detach().cpu().numpy()
        gts = pred * st["std"] + st["mean"]
        STATE["motions"].append({"model": "momask", "gts": gts[0], "fps": 20.0})
    print(f"  {len(STATE['motions'])} animacao(oes) gerada(s)")

STATE["loaded"] = True
if not STATE["motions"]:
    raise RuntimeError("Nenhum arquivo de animacao encontrado - veja o log acima")
print(f"PROMPT: {CFG['prompt']}")
print(f"{len(STATE['motions'])} animacao(oes) prontas -> rode a celula ⑤ EXPORT.")

In [ ]:
# ════════════════════════ EXPORT + PREVIEW + DOWNLOAD ════════════════════════
# Gera GLB / FBX (ossos do Manny) / BVH / NPZ + preview 3D + download
import os, zipfile
import numpy as np
import IPython.display as IP
from google.colab import files

if not STATE.get("loaded") or not STATE["motions"]:
    raise RuntimeError("Rode a celula GENERATE (celula 4) primeiro.")

OUTD = "/content/motion_out"
os.makedirs(OUTD, exist_ok=True)
preview, files_out = [], []

for i, mo in enumerate(STATE["motions"]):
    if mo["model"] == "kimodo":
        b = flatten_motion(read_bvh(mo["bvh"]))
        # BVH do Kimodo (SOMA) vem em centimetros -> detectar e converter p/ metros
        span = float(np.abs(b.local_pos).max())
        scale = 1.0 / 100.0 if span > 5 else 1.0
        if scale != 1.0:
            b.offsets_m = [list(np.array(o, dtype=float) * scale) for o in b.offsets_m]
            b.local_pos = b.local_pos * scale
        # normaliza o root conforme o modo escolhido no CONFIG
        ri = b.root_index
        if CFG["root_mode"] == "floor":
            y0 = b.local_pos[0, ri, 1]
            b.local_pos[:, ri, 1] = CFG["pelvis_h"] + (b.local_pos[:, ri, 1] - y0)
        elif CFG["root_mode"] == "origin":
            b.local_pos[:, ri] = b.local_pos[:, ri] - b.local_pos[0, ri]
        elif CFG["root_mode"] == "inplace":
            b.local_pos[:, ri, 0] = b.local_pos[0, ri, 0]
            b.local_pos[:, ri, 2] = b.local_pos[0, ri, 2]
        mapping = UE5_MAP_SOMA77
    elif mo["model"] == "momask":
        b = gts_to_motion(mo["gts"], root_mode=CFG["root_mode"],
                          target_pelvis_height=CFG["pelvis_h"], fps=mo.get("fps", 20.0))
        mapping = UE5_MAP_SMPL22
    else:  # hymotion
        d = np.load(mo["npz"])
        b = poses_to_motion(d["Rh"], d["poses"], d["trans"],
                            target_pelvis_height=CFG["pelvis_h"], fps=30.0,
                            root_mode=CFG["root_mode"])
        mapping = UE5_MAP_SMPL22

    names = with_ue_names(b, mapping) if CFG["skeleton"] == "ue5_manny" else b.names
    stem = f"{OUTD}/motion_{mo['model']}_{i:02d}"
    write_glb_skeleton(b.names, b.parents, b.local_pos, b.local_quats, b.fps,
                       stem + ".glb", node_names=names, anim_name="motion")
    write_fbx_skeleton(b.names, b.parents, b.local_pos, b.local_quats, b.fps,
                       stem + "_ue.fbx", node_names=names, anim_name="motion")
    write_bvh(b, stem + ".bvh")
    if mo["model"] == "momask":
        np.save(stem + "_raw.npy", mo["gts"])
        files_out.append(stem + "_raw.npy")
    elif mo["model"] == "hymotion":
        d2 = np.load(mo["npz"])
        np.savez(stem + "_raw.npz", **{k: d2[k] for k in d2.files})
        files_out.append(stem + "_raw.npz")
    wp = fk_world_positions(b.local_quats, b.local_pos, b.parents)
    preview.append({"label": f"[{i}] {CFG['prompt'][:52]}",
                    "parents": b.parents, "positions": wp, "fps": b.fps})
    files_out += [stem + ".glb", stem + "_ue.fbx", stem + ".bvh"]

print(f"OSSOS: {CFG['skeleton']} | ROOT: {CFG['root_mode']} | PELVIS: {CFG['pelvis_h']}m")
print("Arquivos:")
for f_ in files_out:
    print("  -", os.path.basename(f_))

# --- Preview 3D (stick figure, arraste p/ girar, barra p/ scrub) ---
IP.display(IP.HTML(build_preview_html(preview, title="AI Motion Preview - Maid Cat Cafe")))

# --- Preview do HY-Motion (HTML oficial, se aplicavel) ---
if STATE.get("hymotion_html"):
    IP.display(IP.HTML(f"<h4>Preview oficial HY-Motion:</h4>{STATE['hymotion_html']}"))

# --- Download (zip com tudo) ---
zpath = "/content/motion_pack.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f_ in files_out:
        z.write(f_)
print("Baixando motion_pack.zip ...")
files.download(zpath)
print("Depois: veja o passo a passo de importacao no UE5 na ultima celula.")

## 🎮 Importando no Unreal Engine 5 (passo a passo)

Os arquivos `*_ue.fbx` já vêm com **ossos batizados com os nomes do Manny**
(`pelvis`, `spine_01..03`, `thigh_l`, `calf_l`, `clavicle_l`, `thumb_01_l` …),
o que deixa o retarget quase automático.

1. **Importar o FBX**
   - Arraste o `motion_xxx_ue.fbx` para o Content Browser.
   - No diálogo de import: deixe **Skeletal Mesh** → **Import** (também marca
     *Import Animation* automaticamente).
   - Isso cria um `SK_*` + uma `AnimSequence` com o esqueleto importado.

2. **Retargetar para o seu Manny (IK Retargeter)**
   - `Tools → Animation → IK Retargeter` (ou *Window → IK Retargeting Tool* em versões recentes).
   - **Source Asset** = a AnimSequence importada · **Target Asset** = seu `SK_Mannequin`
     (ou o esqueleto do seu personagem).
   - Como os nomes já casam, o auto-mapeamento resolve quase tudo; confira a lista
     (pelvis↔pelvis, spine_01↔spine_01, thigh_l↔thigh_l…).
   - **Generate Retargeted Animation** → salve no seu pasta.

3. **Se o personagem flutua / afunda**
   - No CONFIG usei *Root = floor* com pelvis a 0.95 m (altura do Manny). Para um
     personagem mais alto/curto, ajuste *Altura do pelvis (m)* e gere de novo.
   - Alternativas: `origin` (pelvis no Y=0, movimento de raiz fiel) ou
     `inplace` (sem deslocamento — bom para ações fixas, como servir café).

4. **Dedos**
   - Só o **Kimodo (SOMA-77)** anima os dedos (5 dedos × 3-4 ossos, já mapeados
     para `thumb_01_l`, `index_02_r` etc.). HY-Motion e MoMask usam o corpo
     SMPL-22 (mãos em pose neutra).

## ⚠️ Limitações honestas (importante p/ um jogo)
- **Sem objetos**: o modelo não "sabe" onde está a xícara. O braço *finge* pegar.
  No UE, corre com **IK** (pegue a xícara como target da mão) ou use o plugin
  *Epic MetaHuman Animator* (markerless) para capturar a cena real com objetos.
- **Sem looping perfeito**: cada geração é única; para loops, gere e faça fade
  (Blend Space) ou alinhe início/fim manualmente.
- **Animais (gatos)**: nenhum desses modelos gera gato. Fica pra depois
  (keyframes no UE/Cascadeur ou outra solução).
- **Qualidade**: se a variação não ficou boa, mude o **seed** e rode GENERATE
  de novo (o modelo já está carregado — é rápido).

## 💡 Dicas de prompt (em inglês)
- Seja específico e físico: *"a maid tiptoes to the counter and pours tea slowly"*.
- Um movimento por vez funciona melhor que uma sequência longa.
- Para o cafe: "serves", "pours", "bows", "curtsies", "dusts the table",
  "holds a plate with both hands", "steps sideways gracefully".

## 🔄 Rotina de trabalho sugerida
1. CONFIG com 2-4 variações → GENERATE → EXPORT.
2. Teste o preview; gostou? Importe o FBX no UE e retargete.
3. Não gostou? Troque o seed (mesmo modelo, sem download) e gere de novo.
4. Quer outro modelo? `Runtime → Restart`, CONFIG com o outro modelo.
